In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '..')
import re
import numpy
import pyclifford as pc
import mcp_server as mcp 

# MCP Server for PyClifford

## `Server.py`

### Overview

The `server.py` module is the main entry point for the PyClifford MCP server. It defines tools for access the [PyClifford](https://github.com/hongyehu/PyClifford) quantum framework using the [FastMCP 2.0](https://gofastmcp.com/getting-started/welcome) framework. This server facilitates quantum computations by providing endpoints Pauli opertor algebra, stabilizer state manipulation and Clifford circuit simulation.

### Key Functions

#### `pauli_operator_product`

- **Purpose:** Computes the product (matrix multiplication) of two single Pauli terms.
- **Input:** Accepts two `PauliTerm` objects.
- **Output:** Returns a single `PauliTerm` representing the product.
- **Examples:**
  - **Input:** $(+i X_1) (Y_1)$
  - **Output:** $- Z_{1}$
- **Schema:**
  - **Text-based Input:** 
    ```json
    {
      "op1": {"text": "+i X_1"},
      "op2": {"text": "Y_1"}
    }
    ```
  - **Structured Input:**
    ```json
    {
      "op1": {"coefficient": 1j, "pauli_string": {"1": "X"}},
      "op2": {"coefficient": 1, "pauli_string": {"1": "Y"}}
    }
    ```
  - **Output Schema:**
    ```json
    {
      "coefficient": -1,
      "pauli_string": {"1": "Z"},
      "text": "- Z_{1}"
    }
    ```

#### `operator_product`

- **Purpose:** Computes the product (matrix multiplication) of two quantum operators (Pauli polynomials).
- **Input:** Accepts two `Operator` objects, each a sum of Pauli terms.
- **Output:** Returns a new `Operator` representing the product.
- **Examples:**
  - **Input:** $(-0.5 Z_1 + 0.5 Z_2) (X_1)$
  - **Output:** $-0.5i Y_{1} + 0.5 X_{1} Z_{2}$
- **Schema:**
  - **Text-based Input:**
    ```json
    {
      "op1": {"text": "-0.5 Z_1 + 0.5 Z_2"},
      "op2": {"text": "X_1"}
    }
    ```
  - **Structured Input:**
    ```json
    {
      "op1": {
        "terms": [
          {"coefficient": -0.5, "pauli_string": {"1": "Z"}},
          {"coefficient": 0.5, "pauli_string": {"2": "Z"}}
        ]
      },
      "op2": {
        "terms": [
          {"coefficient": 1, "pauli_string": {"1": "X"}}
        ]
      }
    }
    ```
  - **Output Schema:**
    ```json
    {
      "terms": [
        {"coefficient": 0.5, "pauli_string": {"1": "X", "2": "Z"}, "text": "0.5 X_{1} Z_{2}"},
        {"coefficient": -0.5j, "pauli_string": {"1": "Y"}, "text": "-0.5i Y_{1}"},
      ],
      "text": "0.5 X_{1} Z_{2} - 0.5i Y_{1}"
    }
    ```

In [31]:
op1 = mcp.Operator(text="-0.5 Z_1 + 0.5 Z_2")
op2 =mcp.Operator(text="X_1")
op = mcp.operator_product(op1,op2)
print(f'({op1.text})  ({op2.text}) = {op.text}')

(-0.5 Z_{1} + 0.5 Z_{2})  (X_{1}) = 0.5 X_{1} Z_{2} - 0.5i Y_{1}


### Running the Server

- The server is initiated by running `python3 mcp_server/server.py`, which starts the FastMCP server for handling requests related to quantum operator products. 
- Install the [uv](https://docs.astral.sh/uv/) package by `curl -sSf https://astral.sh/uv/install.sh | bash`.
- Add the following to `.cursor/mcp.json` to enable CursorAI client to run PyClifford MCP server.
```json
    "pyclifford": {
        "name": "pyclifford",                    // Server identifier
        "version": "0.1.0",                      // Server version
        "description": "MCP server to use PyClifford package",
        "command": "uv",                         // Package manager to use
        "args": [
            "run",                               // Run command
            "--with", "mcp[cli]",                // Include MCP CLI tools
            "mcp", "run",                        // MCP commands
            "[/path/to...]/mcp_server/server.py" // Server script path
        ]
    }
```


## `Schemas.py`

### Overview

The `schemas.py` module provides robust, AI-friendly [Pydantic models](https://gofastmcp.com/servers/tools#pydantic-models) for representing, parsing, and serializing PyClifford objects in a variety of notations. It is designed to support both structured and text-based input. FastMCP automatically parses Pydantic models to generate JSON schemas, enabling AI agents to understand and interact with tools effectively.

Pydantic models serve as a bridge for three-way communication across different endpoints:
1. **Human**: Prefers flexible and readable text-based inputs—including mathematical expressions and language descriptions—for operators, states, and quantum circuits.
2. **AI Agent**: Follows structured JSON schemas that define the expected input/output formats for MCP tools, enabling MCP server to *validate* AI-generated inputs for correctness and provide *semantic guidance* to help the agent construct meaningful requests.
3. **Python**: Provides PyClifford objects, implementing efficient algorithms for Pauli algebra and Clifford circuit simulation—realizing the core functionality of quantum simulation.

### Key Classes

#### `PauliTerm`
- **Purpose:** Represents a single Pauli monomial (a complex scalar coefficient times a tensor product of single-qubit Pauli operators).
- **Aliases:** Also known as "PauliMonomial" (PyClifford), "PauliString" (Cirq), or "Pauli" (Qiskit).
- **Input Flexibility:**  
  - Accepts structured input via `coefficient` and `pauli_string` fields.
  - Accepts text-based input via the `text` field, which is automatically parsed.
- **Parsing:**  
  - Supports standard, compact, and LaTeX-style notations.
    - **Standard Notation:** e.g., `+i X_1 Y_2`, `-0.5 Z_1`
    - **Compact Notation:** e.g., `-iIXYIZ`, `X1Y2Z5`
    - **LaTeX Notation:** e.g., `1.2*\sigma^{01203}`, `0.5 \sigma_0^3 \sigma_1^2`
  - Handles coefficients as real, complex, or phase factors (e.g., `+i`, `-1`, `0.5`, `1+2j`).
    - **Complex Coefficients:** Handles forms like `-(0.5+i)`, `0.5+i`, `-1j`, etc. Correctly splits terms even when `+` or `-` appear inside parentheses of complex numbers.
- **AI Agent Note:**  
  - If unsure about parsing, simply provide the full term as a string to the `text` field; the model will handle parsing.


In [3]:
exprs = [
    '- X_1 Z_{2} Y3 X{4} Z(5)',
    '+iI XZ YXZ',
    '(-0.5+i)\\sigma^{01z2x3}',
    '2.6*\\sigma_0^{0} \\sigma_{1}^{1}\\sigma_2^3 \\sigma_3^y\\sigma^{X}_4\\sigma^3_{5}',
    '1'
]
for expr in exprs:
    print(mcp.PauliTerm(text=expr))

coefficient=(-1+0j) pauli_string={1: 'X', 2: 'Z', 3: 'Y', 4: 'X', 5: 'Z'} text='- X_{1} Z_{2} Y_{3} X_{4} Z_{5}'
coefficient=1j pauli_string={1: 'X', 2: 'Z', 3: 'Y', 4: 'X', 5: 'Z'} text='+i X_{1} Z_{2} Y_{3} X_{4} Z_{5}'
coefficient=(-0.5+1j) pauli_string={1: 'X', 2: 'Z', 3: 'Y', 4: 'X', 5: 'Z'} text='(-0.5+i) X_{1} Z_{2} Y_{3} X_{4} Z_{5}'
coefficient=(2.6+0j) pauli_string={1: 'X', 2: 'Z', 3: 'Y', 4: 'X', 5: 'Z'} text='2.6 X_{1} Z_{2} Y_{3} X_{4} Z_{5}'
coefficient=(1+0j) pauli_string={} text='I'


#### `Operator`
- **Purpose:** Represents a generic quantum operator as a linear combination (sum) of Pauli terms.
- **Aliases:** Also known as "PauliPolynomial" (PyClifford), "PauliSum" (Cirq), or "PauliSumOp" (Qiskit).
- **Input Flexibility:**  
  - Accepts a list of `PauliTerm` objects via the `terms` field.
  - Accepts a full operator expression as a string via the `text` field, which is parsed into terms.
- **Parsing:**  
  - Splits the operator string into terms, correctly handling `+`/`-` signs, even inside complex numbers or parentheses.
  - Each term is parsed using the same flexible logic as `PauliTerm`.
- **AI Agent Note:**  
  - If you have a long or complex operator expression, or are unsure about the structure, provide it as a string to the `text` field.

In [4]:
exprs = [
    "+i X_1 Y_2 -0.5 Z_1 + 0.5 Z_2",
    "-X1Y2Z3+Y1Z2-X3",
    "+iX_1Z_{2}Y3X{4}Z5 - (0.5+i) X_1 + 0.5 Y_2",
    "1.2*\\sigma^{01203} - 0.5*X_1Y_2"
]
for expr in exprs:
    print(mcp.Operator(text=expr))

terms=[PauliTerm(coefficient=1j, pauli_string={1: 'X', 2: 'Y'}, text='+i X_{1} Y_{2}'), PauliTerm(coefficient=(-0.5+0j), pauli_string={1: 'Z'}, text='-0.5 Z_{1}'), PauliTerm(coefficient=(0.5+0j), pauli_string={2: 'Z'}, text='0.5 Z_{2}')] text='+i X_{1} Y_{2} - 0.5 Z_{1} + 0.5 Z_{2}'
terms=[PauliTerm(coefficient=(-1+0j), pauli_string={1: 'X', 2: 'Y', 3: 'Z'}, text='- X_{1} Y_{2} Z_{3}'), PauliTerm(coefficient=(1+0j), pauli_string={1: 'Y', 2: 'Z'}, text='Y_{1} Z_{2}'), PauliTerm(coefficient=(-1+0j), pauli_string={3: 'X'}, text='- X_{3}')] text='- X_{1} Y_{2} Z_{3} + Y_{1} Z_{2} - X_{3}'
terms=[PauliTerm(coefficient=1j, pauli_string={1: 'X', 2: 'Z', 3: 'Y', 4: 'X', 5: 'Z'}, text='+i X_{1} Z_{2} Y_{3} X_{4} Z_{5}'), PauliTerm(coefficient=(-0.5-1j), pauli_string={1: 'X'}, text='(-0.5-i) X_{1}'), PauliTerm(coefficient=(0.5+0j), pauli_string={2: 'Y'}, text='0.5 Y_{2}')] text='+i X_{1} Z_{2} Y_{3} X_{4} Z_{5} + (-0.5-i) X_{1} + 0.5 Y_{2}'
terms=[PauliTerm(coefficient=(1.2+0j), pauli_string={1:

#### `CliffordUnitary`
- **Purpose:** Represents a Clifford unitary transformation specified by its action on Pauli operators.
- **Mathematical Definition:** A Clifford unitary $U$ is defined by its transformation (conjugation action) on any operator $O$ as $O \rightarrow U^\dagger O U$.
- **Key Properties:**
  - The action of $U$ is fully specified by its effect on the Pauli group generators (e.g., $X_i$, $Z_i$ for $i=1,...,N$).
  - The forward map is a dictionary mapping each generator to its image under unitary transformation.
- **Key Features:**
  - **Phase Handling:** Automatically handles phase factors in the mapping.
  - **Validation:** Ensures each key contains exactly one single-qubit Pauli operator (X or Z).
  - **Conversion:** Can be converted to/from PyClifford's `CliffordGate` and `CliffordMap` objects.

- **AI Agent Note:**
  - The `clifford_map` field should be a dictionary where keys are string representations of Pauli group generators.
  - The `text` field provides a human-readable description of the Clifford unitary.
  - This model is useful for representing, serializing, and communicating Clifford unitaries in a structured way.


In [69]:
expr = """
  X2-> + X2 X4
  Z2-> + Z2
  X4-> + X4
  Z4-> + Z2 Z4
"""
print(mcp.CliffordUnitary(text=expr))

clifford_map={'X_{2}': PauliTerm(coefficient=(1+0j), pauli_string={2: 'X', 4: 'X'}, text='X_{2} X_{4}'), 'Z_{2}': PauliTerm(coefficient=(1+0j), pauli_string={2: 'Z'}, text='Z_{2}'), 'X_{4}': PauliTerm(coefficient=(1+0j), pauli_string={4: 'X'}, text='X_{4}'), 'Z_{4}': PauliTerm(coefficient=(1+0j), pauli_string={2: 'Z', 4: 'Z'}, text='Z_{2} Z_{4}')} text='X_{2} -> X_{2} X_{4}, Z_{2} -> Z_{2}, X_{4} -> X_{4}, Z_{4} -> Z_{2} Z_{4}'


In [70]:
mcp.CliffordUnitary(text=expr).to_obj().forward_map

CliffordMap(
  X2-> + X2 X4
  Z2-> + Z2
  X4-> + X4
  Z4-> + Z2 Z4)


### PyClifford Support

- Designed for use with PyClifford quantum framework.
- Every Pydantic model provides `.to_obj()` and `.from_obj()` methods for conversion to and from PyClifford objects.
